In [1]:
# %pip install python-dotenv
# %uv add dspy
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

## check aicodetools library

In [16]:
from gepa_setup import tlm, os , dspy , lm ,bench, sb_metas, time, threading, tiktoken, deque, BaseCallback, GEPAState, base_program

In [22]:
base_program.named_predictors()

[('react.react',
  Predict(StringSignature(query, github_repo, git_commit, trajectory -> next_thought, next_tool_name, next_tool_args
      instructions='Solve the question and provide the answer in the correct format.\n\nYou are an Agent. In each episode, you will be given the fields `query`, `github_repo`, `git_commit` as input. And you can see your past trajectory so far.\nYour goal is to use one or more of the supplied tools to collect any necessary information for producing `result`.\n\nTo do this, you will interleave next_thought, next_tool_name, and next_tool_args in each turn, and also when finishing the task.\nAfter each tool call, you receive a resulting observation, which gets appended to your trajectory.\n\nWhen writing next_thought, you may reason about the current situation and plan for future steps.\nWhen selecting the next_tool_name and its next_tool_args, the tool must be one of:\n\n(1) read_file, whose description is <desc>          Read a text file with optional line

In [17]:
state = GEPAState.load('runs/gepa-state-with-gold')

In [24]:
def idxmax(lst):
    """Return the index of the maximum value in a list."""
    max_val = max(lst)
    return lst.index(max_val)


gepa_state = state
best_prog_idx = idxmax(gepa_state.per_program_tracked_scores)
best_progs = gepa_state.program_candidates
best_prog = best_progs[best_prog_idx]

optimized_program = best_prog
latest_prog = best_progs[-1]

In [25]:
len(best_progs)

9

In [20]:
best_prog_idx

2

In [ ]:
import dspy
import os
import json
import dspy

In [ ]:
def metric_aggregator(evaluate_output):
    """
    Aggregates:
        - overall accuracy
        - mean submitted
        - mean output_match
        - mean landmarks
    """

    overall_accuracy, detailed_results, score_objects = evaluate_output

    ntotal = len(score_objects)

    if ntotal == 0:
        return {
            "overall_accuracy": 0.0,
            "mean_submitted": 0.0,
            "mean_output_match": 0.0,
            "mean_landmarks": 0.0,
            "ntotal": 0,
        }

    total_submitted = 0.0
    total_output_match = 0.0
    total_landmarks = 0.0

    for score_obj in score_objects:
        score_dict = score_obj.score_dict

        total_submitted += score_dict.get("submitted", 0.0)
        total_output_match += score_dict.get("output_match", 0.0)
        total_landmarks += score_dict.get("landmarks", 0.0)

    return {
        "overall_accuracy": overall_accuracy,
        "mean_submitted": round(total_submitted / ntotal, 4),
        "mean_output_match": round(total_output_match / ntotal, 4),
        "mean_landmarks": round(total_landmarks / ntotal, 4),
        "ntotal": ntotal,
    }

In [ ]:

base_dir = "runs/progress"
os.makedirs(base_dir, exist_ok=True)

output_jsonl_path = os.path.join(base_dir, "iter-results.jsonl")

for i, curr_prog in enumerate(best_progs):

    print(f"\n========== Evaluating Program {i} ==========\n")

    # Create per-program directory
    program_dir = os.path.join(base_dir, f"program_{i}")
    os.makedirs(program_dir, exist_ok=True)

    evaluator = dspy.Evaluate(
        devset=bench.test_set,
        metric=sb_metas[0].metric,
        num_threads=8,
        display_table=True,
        display_progress=True,
        max_errors=100 * len(bench.test_set),
        provide_traceback=True,
        failure_score=0,
        return_all_scores=True,
        return_outputs=True,
        save_as_json=os.path.join(program_dir, "optimized_react.json"),
        save_as_csv=os.path.join(program_dir, "optimized_react.csv"),
    )

    # Run evaluation
    results = evaluator(curr_prog)

    # Aggregate metric
    complete_result = metric_aggregator(results)

    # Extract instructions from predictors
    prog_instructions = {}

    for name, pred in curr_prog.named_predictors():
        prog_instructions[name] = pred.signature.instructions

    # Prepare record
    record = {
        "program_index": i,
        "prog_instructions": prog_instructions,
        "complete_result": complete_result,
    }

    # Append to JSONL
    with open(output_jsonl_path, "a") as f:
        f.write(json.dumps(record) + "\n")

    print(f"Saved results for program {i}")
